# Training Metrics and Hardware Utilization Visualization

This notebook loads `metrics.json` and creates interactive Plotly charts to analyze model convergence and hardware utilization of your RTX 4070 GPU during training.

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load the metrics data robustly
import os
metrics_file = "metrics.json"
if not os.path.exists(metrics_file):
    # Try relative path from workspace root
    metrics_file = os.path.join("chapter_1_transformers", "basic_transformer_implementation", "metrics.json")

with open(metrics_file, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

## 1. Machine Learning Metrics (Loss, Perplexity, Accuracy)

In [ ]:
# Plot Train Loss and Dev Accuracy
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['train_loss'], name="Train Loss", mode="lines+markers", line=dict(color="royalblue", width=3)),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['val_acc'] * 100, name="Validation Accuracy", mode="lines+markers", line=dict(color="forestgreen", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="Training Loss and Validation Accuracy Over Epochs",
    xaxis_title="Epoch",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)

fig.update_yaxes(title_text="Cross Entropy Loss", secondary_y=False)
fig.update_yaxes(title_text="Accuracy (%)", secondary_y=True)

fig.show()

## 2. Hardware Utilization (Peak VRAM Allocated vs. Reserved)

VRAM allocation tracking is crucial in LLM engineering. 
- **Allocated Memory**: The VRAM actively holding tensors.
- **Reserved Memory**: The VRAM cached by PyTorch's memory allocator (caching allocator) to avoid the high overhead of repeatedly querying the CUDA driver.

In [ ]:
fig_vram = go.Figure()
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_allocated_mb'],
    name='Peak VRAM Allocated (MB)',
    marker_color='crimson'
))
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_reserved_mb'],
    name='Peak VRAM Reserved (MB)',
    marker_color='lightcoral'
))

fig_vram.update_layout(
    barmode='group',
    title_text='Peak GPU VRAM Usage (Allocated vs. Reserved)',
    xaxis_title='Epoch',
    yaxis_title='VRAM (MB)',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_vram.show()

## 3. Data Processing Rate (Throughput vs. Goodput)

- **Throughput**: Total raw tokens (characters) processed per second.
- **Goodput**: Useful tokens processed per second (excluding padding).

### Why do the Throughput and Goodput lines overlap exactly?
In this character-level implementation, we slice the text corpus into fixed chunks of 20 characters directly. **No padding tokens (`<pad>`) are used.** Since $100\%$ of the processed tokens are useful learning tokens, **Goodput is mathematically identical to Throughput**, resulting in the two lines overlapping perfectly on the chart.

In [ ]:
fig_tp = go.Figure()
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['throughput_tokens_sec'],
    name='Throughput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='darkorange', width=3)
))
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['goodput_tokens_sec'],
    name='Goodput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='orange', width=2, dash='dash')
))

fig_tp.update_layout(
    title_text='Data Processing Rate (Throughput vs. Goodput)',
    xaxis_title='Epoch',
    yaxis_title='Tokens / Second',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_tp.show()

## 4. Compute Performance (TFLOPs and Model FLOPs Utilization)

- **Achieved TFLOPs/sec**: The absolute processing rate of raw floating point operations.
- **Model FLOPs Utilization (MFU %)**: The ratio of achieved compute performance to the hardware's peak theoretical performance. For your RTX 4070 Super, peak dense FP16 tensor core throughput is **142.2 TFLOPs/sec**.

In [ ]:
# Plot Achieved TFLOPs/sec
fig_tflops = go.Figure()
fig_tflops.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['tflops_per_sec'], 
    name="Achieved TFLOPs/sec", 
    mode="lines+markers", 
    line=dict(color="mediumpurple", width=3)
))
fig_tflops.update_layout(
    title_text="Achieved Compute Performance (TFLOPs/sec)",
    xaxis_title="Epoch",
    yaxis_title="TFLOPs / Second",
    template="plotly_dark"
)
fig_tflops.show()

# Plot MFU (%)
fig_mfu = go.Figure()
fig_mfu.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['mfu_percent'], 
    name="MFU (%)", 
    mode="lines+markers", 
    line=dict(color="hotpink", width=3)
))
fig_mfu.update_layout(
    title_text="Model FLOPs Utilization (MFU %)",
    xaxis_title="Epoch",
    yaxis_title="MFU (%)",
    template="plotly_dark"
)
fig_mfu.show()